# Strict Single-Source Alt-Data Ablation

This notebook reruns single-ticker experiments with stricter methodology.

Tickers:
- `TSLA`
- `AAPL`

Feature sets:
- `price_only`
- `price_plus_google_trends`
- `price_plus_gdelt`
- `price_plus_reddit_sentiment`
- `price_plus_reddit_attention`

For each alt-data source, the feature block uses:
- `t-1`
- `t-2`
- `rolling 3d`
- `rolling 5d`
- `surprise vs rolling baseline`

Models:
- `logreg`
- `svm_rbf`
- `random_forest`
- `hist_gradient_boosting`
- `mlp_small`

Methodology additions:
- strict time alignment: all alt-data shifted by one trading day before transforms
- threshold tuning for `svm_rbf` and `mlp_small` using train/validation only
- block-bootstrap confidence intervals for balanced accuracy and ROC AUC
- DeLong test for AUC vs `price_only`
- walk-forward fold metrics and fold-by-fold deltas vs `price_only`

Artifacts are written to `notebooks_two_tickers/outputs/strict_source_ablation`.


In [1]:
import pandas as pd

from strict_source_ablation_experiment import run_strict_source_ablation_experiment

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 320)


In [2]:
result = run_strict_source_ablation_experiment()

print(f"Dataset: {result['dataset_path']}")
print(f"Metrics: {result['metrics_path']}")
print(f"Predictions: {result['predictions_path']}")
print(f"Threshold tuning: {result['tuning_path']}")
print(f"Bootstrap summary: {result['bootstrap_path']}")
print(f"Fold metrics: {result['cv_fold_path']}")
print(f"Fold deltas: {result['cv_delta_path']}")
print(f"DeLong summary: {result['delong_path']}")
print(f"Metadata: {result['metadata_path']}")


Dataset: C:\Users\user\OneDrive\Documents\magisterka\praca magisterska\code\notebooks_two_tickers\outputs\datasets\stock_direction_two_tickers_base.csv
Metrics: C:\Users\user\OneDrive\Documents\magisterka\praca magisterska\code\notebooks_two_tickers\outputs\strict_source_ablation\metrics.csv
Predictions: C:\Users\user\OneDrive\Documents\magisterka\praca magisterska\code\notebooks_two_tickers\outputs\strict_source_ablation\predictions.csv
Threshold tuning: C:\Users\user\OneDrive\Documents\magisterka\praca magisterska\code\notebooks_two_tickers\outputs\strict_source_ablation\threshold_tuning.csv
Bootstrap summary: C:\Users\user\OneDrive\Documents\magisterka\praca magisterska\code\notebooks_two_tickers\outputs\strict_source_ablation\bootstrap_summary.csv
Fold metrics: C:\Users\user\OneDrive\Documents\magisterka\praca magisterska\code\notebooks_two_tickers\outputs\strict_source_ablation\cv_fold_metrics.csv
Fold deltas: C:\Users\user\OneDrive\Documents\magisterka\praca magisterska\code\note

In [3]:
test_metrics = result["metrics"].query("split == 'test'").copy()
test_metrics = test_metrics.sort_values(["ticker", "model_name", "feature_set"]).reset_index(drop=True)
test_metrics


,ticker,model_name,feature_set,split,n_rows,n_features,train_rows_per_feature,decision_threshold,threshold_tuned,validation_rows,validation_balanced_accuracy,accuracy,balanced_accuracy,f1,roc_auc
0,AAPL,hist_gradient_boosting,price_only,test,128,14,44.500000,0.50,False,0,NaN,0.546875,0.533990,0.618421,0.552709
1,AAPL,hist_gradient_boosting,price_plus_gdelt,test,128,29,21.482759,0.50,False,0,NaN,0.492188,0.492857,0.511278,0.541872
2,AAPL,hist_gradient_boosting,price_plus_google_trends,test,128,19,32.789474,0.50,False,0,NaN,0.531250,0.515271,0.615385,0.540148
3,AAPL,hist_gradient_boosting,price_plus_reddit_attention,test,128,34,18.323529,0.50,False,0,NaN,0.523438,0.511084,0.596026,0.525616
4,AAPL,hist_gradient_boosting,price_plus_reddit_sentiment,test,128,34,18.323529,0.50,False,0,NaN,0.593750,0.590148,0.628571,0.595567
5,AAPL,logreg,price_only,test,128,14,44.500000,0.50,False,0,NaN,0.507812,0.511576,0.511628,0.497044
6,AAPL,logreg,price_plus_gdelt,test,128,29,21.482759,0.50,False,0,NaN,0.468750,0.455172,0.552632,0.490394
7,AAPL,logreg,price_plus_google_trends,test,128,19,32.789474,0.50,False,0,NaN,0.531250,0.533005,0.545455,0.535961
8,AAPL,logreg,price_plus_reddit_attention,test,128,34,18.323529,0.50,False,0,NaN,0.515625,0.506897,0.575342,0.503695
9,AAPL,logreg,price_plus_reddit_sentiment,test,128,34,18.323529,0.50,False,0,NaN,0.500000,0.498522,0.529412,0.505911


In [4]:
summary = test_metrics.pivot_table(
    index=["ticker", "model_name"],
    columns="feature_set",
    values=["balanced_accuracy", "roc_auc"],
)
summary


balanced_accuracy                                                                                                      roc_auc                                                                                                  
feature_set                          price_only price_plus_gdelt price_plus_google_trends price_plus_reddit_attention price_plus_reddit_sentiment price_only price_plus_gdelt price_plus_google_trends price_plus_reddit_attention price_plus_reddit_sentiment
ticker model_name                                                                                                                                                                                                                                             
AAPL   hist_gradient_boosting          0.533990         0.492857                 0.515271                    0.511084                    0.590148   0.552709         0.541872                 0.540148                    0.525616                    0.595567
       logreg                          0.511576         0.455172                 0.533005                    0.506897                    0.498522   0.497044         0.490394                 0.535961                    0.503695                    0.505911
       mlp_small                       0.521429         0.528079                 0.517734                    0.504187                    0.454187   0.544828         0.464286                 0.523892                    0.469212                    0.469212
       random_forest                   0.505665         0.567241                 0.474138                    0.532759                    0.581773   0.538177         0.559113                 0.522167                    0.507143                    0.561576
       svm_rbf                         0.471921         0.449015                 0.492857                    0.431527                    0.499261   0.475000         0.452833                 0.485961                    0.428079                    0.491256
TSLA   hist_gradient_boosting          0.581675         0.576762                 0.463277                    0.545443                    0.503070   0.569885         0.572095                 0.507492                    0.613854                    0.488578
       logreg                          0.609555         0.542864                 0.624048                    0.596168                    0.574429   0.618767         0.540162                 0.628592                    0.591255                    0.581921
       mlp_small                       0.634733         0.491525                 0.481823                    0.507246                    0.514493   0.650209         0.344141                 0.416851                    0.664210                    0.534758
       random_forest                   0.625276         0.573078                 0.590273                    0.565954                    0.596168   0.611152         0.571850                 0.606485                    0.591501                    0.573815
       svm_rbf                         0.604520         0.454802                 0.511791                    0.500000                    0.500000   0.607222         0.515352                 0.581552                    0.469049                    0.438958

In [5]:
bootstrap_summary = result["bootstrap_summary"].copy()
bootstrap_summary = bootstrap_summary.sort_values(["ticker", "model_name", "feature_set"]).reset_index(drop=True)
bootstrap_summary


,ticker,model_name,feature_set,split,decision_threshold,balanced_accuracy_ci_low,balanced_accuracy_ci_high,roc_auc_ci_low,roc_auc_ci_high
0,AAPL,hist_gradient_boosting,price_only,test,0.50,0.447101,0.625798,0.443673,0.650605
1,AAPL,hist_gradient_boosting,price_plus_gdelt,test,0.50,0.415256,0.571469,0.459445,0.639979
2,AAPL,hist_gradient_boosting,price_plus_google_trends,test,0.50,0.421865,0.613139,0.429170,0.642771
3,AAPL,hist_gradient_boosting,price_plus_reddit_attention,test,0.50,0.437690,0.581879,0.428142,0.643613
4,AAPL,hist_gradient_boosting,price_plus_reddit_sentiment,test,0.50,0.505594,0.655928,0.491152,0.674205
5,AAPL,logreg,price_only,test,0.50,0.429535,0.592218,0.396055,0.604870
6,AAPL,logreg,price_plus_gdelt,test,0.50,0.369895,0.541896,0.378877,0.602103
7,AAPL,logreg,price_plus_google_trends,test,0.50,0.468990,0.614649,0.425585,0.652830
8,AAPL,logreg,price_plus_reddit_attention,test,0.50,0.418829,0.593590,0.396339,0.609638
9,AAPL,logreg,price_plus_reddit_sentiment,test,0.50,0.422278,0.580122,0.408588,0.604007


In [6]:
delong_summary = result["delong_summary"].copy()
delong_summary = delong_summary.sort_values(["ticker", "model_name", "feature_set_alt"]).reset_index(drop=True)
delong_summary


,ticker,model_name,feature_set_alt,auc_a,auc_b,auc_diff,auc_diff_se,z_stat,p_value
0,AAPL,hist_gradient_boosting,price_plus_gdelt,0.552709,0.541872,0.010837,0.049747,0.217851,0.827545
1,AAPL,hist_gradient_boosting,price_plus_google_trends,0.552709,0.540148,0.012562,0.028816,0.435918,0.662896
2,AAPL,hist_gradient_boosting,price_plus_reddit_attention,0.552709,0.525616,0.027094,0.050406,0.537511,0.590915
3,AAPL,hist_gradient_boosting,price_plus_reddit_sentiment,0.552709,0.595567,-0.042857,0.053298,-0.804100,0.421339
4,AAPL,logreg,price_plus_gdelt,0.497044,0.490394,0.006650,0.037297,0.178306,0.858482
5,AAPL,logreg,price_plus_google_trends,0.497044,0.535961,-0.038916,0.032861,-1.184267,0.236307
6,AAPL,logreg,price_plus_reddit_attention,0.497044,0.503695,-0.006650,0.037845,-0.175725,0.860510
7,AAPL,logreg,price_plus_reddit_sentiment,0.497044,0.505911,-0.008867,0.043389,-0.204360,0.838072
8,AAPL,mlp_small,price_plus_gdelt,0.544828,0.464286,0.080542,0.075238,1.070499,0.284395
9,AAPL,mlp_small,price_plus_google_trends,0.544828,0.523892,0.020936,0.054891,0.381411,0.702898


In [7]:
cv_delta = result["cv_fold_deltas"].copy()
cv_delta_summary = cv_delta.groupby(["ticker", "model_name", "feature_set_alt"], as_index=False)[["delta_balanced_accuracy", "delta_roc_auc"]].mean()
cv_delta_summary = cv_delta_summary.sort_values(["ticker", "model_name", "feature_set_alt"]).reset_index(drop=True)
cv_delta_summary


,ticker,model_name,feature_set_alt,delta_balanced_accuracy,delta_roc_auc
0,AAPL,hist_gradient_boosting,price_plus_gdelt,-0.013507,-0.037649
1,AAPL,hist_gradient_boosting,price_plus_google_trends,0.010570,0.007886
2,AAPL,hist_gradient_boosting,price_plus_reddit_attention,0.007747,-0.018464
3,AAPL,hist_gradient_boosting,price_plus_reddit_sentiment,-0.011155,-0.031401
4,AAPL,logreg,price_plus_gdelt,-0.006858,-0.004220
5,AAPL,logreg,price_plus_google_trends,0.004543,-0.008838
6,AAPL,logreg,price_plus_reddit_attention,-0.040005,-0.048646
7,AAPL,logreg,price_plus_reddit_sentiment,0.010776,0.035969
8,AAPL,mlp_small,price_plus_gdelt,-0.034325,-0.008590
9,AAPL,mlp_small,price_plus_google_trends,0.037181,0.062075


In [8]:
thresholds = result["threshold_tuning"].copy()
thresholds = thresholds.sort_values(["ticker", "model_name", "feature_set"]).reset_index(drop=True)
thresholds


,ticker,model_name,feature_set,split_scope,decision_threshold,threshold_tuned,validation_rows,validation_balanced_accuracy
0,AAPL,hist_gradient_boosting,price_only,final_test,0.50,False,0,NaN
1,AAPL,hist_gradient_boosting,price_plus_gdelt,final_test,0.50,False,0,NaN
2,AAPL,hist_gradient_boosting,price_plus_google_trends,final_test,0.50,False,0,NaN
3,AAPL,hist_gradient_boosting,price_plus_reddit_attention,final_test,0.50,False,0,NaN
4,AAPL,hist_gradient_boosting,price_plus_reddit_sentiment,final_test,0.50,False,0,NaN
5,AAPL,logreg,price_only,final_test,0.50,False,0,NaN
6,AAPL,logreg,price_plus_gdelt,final_test,0.50,False,0,NaN
7,AAPL,logreg,price_plus_google_trends,final_test,0.50,False,0,NaN
8,AAPL,logreg,price_plus_reddit_attention,final_test,0.50,False,0,NaN
9,AAPL,logreg,price_plus_reddit_sentiment,final_test,0.50,False,0,NaN
